# MVP pipeline imágenes de repuestos (DONREP)

## Etapa 0 — Carga de datos

Librerias 

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path


In [2]:
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd

INPUT_XLSX = ROOT / "input" / "products.xlsx"
COL_REF, COL_BRAND, COL_NAME = "Ref Proveedor", "Marca", "Nombre"

df = pd.read_excel(INPUT_XLSX, dtype={COL_REF: str})
df = df[[COL_REF, COL_BRAND, COL_NAME]].copy()
for col in df.columns:
    df[col] = df[col].astype(str).str.strip()
df = df[df[COL_REF].notna() & (df[COL_REF] != "") & (df[COL_REF] != "nan")].reset_index(drop=True)

print(f"{len(df)} productos cargados")
df.head(10)

281 productos cargados


,Ref Proveedor,Marca,Nombre
0,8550504748,MOTRIO,10W30 MOTRIO 1LX12UN
1,8550504747,MOTRIO,10W30 MOTRIO 4LX4UND
2,7711737740,MOTRIO,10W30 MOTRIO 4X4 UNID
3,8550504765,MOTRIO,10W40 MOTRIO A3/B4 SS 1LX12UND
4,8550504764,MOTRIO,10W40 MOTRIO A3/B4 SS 4LX4UND
5,8550504745,MOTRIO,15W40 MOTRIO 1LX12UN
6,7711737737,MOTRIO,15W40 MOTRIO 4LX4UND
7,8550504744,MOTRIO,15W40 MOTRIO 4LX4UND
8,7711649209,RENAULT,185/65R15 ENERGY XM2
9,8550504751,MOTRIO,20W50 MOTRIO 1LX12UN


## Etapa 1 — Normalización de nombres

Es importante revisar visualmente `nombre_original` vs `nombre_limpio` / `termino_busqueda` luego de la normalizacion. De ser necesario se puede ajustar `config/abbreviations.py` .

In [3]:
from pipeline.normalize import normalize_df

df_norm = normalize_df(df, col_nombre=COL_NAME, col_marca=COL_BRAND)

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", 300)
df_norm[["nombre_original", "nombre_limpio", "termino_busqueda"]].head(10)

,nombre_original,nombre_limpio,termino_busqueda
0,10W30 MOTRIO 1LX12UN,10W30 MOTRIO 1L,10W30 MOTRIO 1L
1,10W30 MOTRIO 4LX4UND,10W30 MOTRIO 4L,10W30 MOTRIO 4L
2,10W30 MOTRIO 4X4 UNID,10W30 MOTRIO 4X4 UND,10W30 MOTRIO 4X4 UND
3,10W40 MOTRIO A3/B4 SS 1LX12UND,10W40 MOTRIO A3/B4 SS 1L,10W40 MOTRIO A3/B4 SS 1L
4,10W40 MOTRIO A3/B4 SS 4LX4UND,10W40 MOTRIO A3/B4 SS 4L,10W40 MOTRIO A3/B4 SS 4L
5,15W40 MOTRIO 1LX12UN,15W40 MOTRIO 1L,15W40 MOTRIO 1L
6,15W40 MOTRIO 4LX4UND,15W40 MOTRIO 4L,15W40 MOTRIO 4L
7,15W40 MOTRIO 4LX4UND,15W40 MOTRIO 4L,15W40 MOTRIO 4L
8,185/65R15 ENERGY XM2,185/65R15 ENERGY XM2,185/65R15 ENERGY XM2 RENAULT
9,20W50 MOTRIO 1LX12UN,20W50 MOTRIO 1L,20W50 MOTRIO 1L


In [4]:
# columnas estructuradas extraídas
df_norm[["nombre_original", "pack", "litraje", "pack_ambiguo", "viscosidad", "medida_llanta", "vehiculo_code_raw"]].head(10)

,nombre_original,pack,litraje,pack_ambiguo,viscosidad,medida_llanta,vehiculo_code_raw
0,10W30 MOTRIO 1LX12UN,1LX12UN,1L,False,10W30,,
1,10W30 MOTRIO 4LX4UND,4LX4UND,4L,False,10W30,,
2,10W30 MOTRIO 4X4 UNID,4X4 UND,,True,10W30,,UND
3,10W40 MOTRIO A3/B4 SS 1LX12UND,1LX12UND,1L,False,10W40,,
4,10W40 MOTRIO A3/B4 SS 4LX4UND,4LX4UND,4L,False,10W40,,
5,15W40 MOTRIO 1LX12UN,1LX12UN,1L,False,15W40,,
6,15W40 MOTRIO 4LX4UND,4LX4UND,4L,False,15W40,,
7,15W40 MOTRIO 4LX4UND,4LX4UND,4L,False,15W40,,
8,185/65R15 ENERGY XM2,,,False,,185/65R15,XM2
9,20W50 MOTRIO 1LX12UN,1LX12UN,1L,False,20W50,,


In [5]:
df_norm[["nombre_original", "pack", "litraje", "pack_ambiguo", "viscosidad", "medida_llanta", "vehiculo_code_raw"]].tail(10)

,nombre_original,pack,litraje,pack_ambiguo,viscosidad,medida_llanta,vehiculo_code_raw
271,VIDRIO LUNETA TR SA3,,,False,,,SA3
272,VIDRIO MOVIL PUERTA NDU,,,False,,,NDU
273,VIDRIO MOVIL PUERTA TRASERA DERECHA X52,,,False,,,X52
274,VIDRIO MOVIL PUERTA TRASERA IZQUIER GKO,,,False,,,GKO
275,VIDRIO PARABRISA NDU,,,False,,,NDU
276,VIDRIO PARABRISAS NKG,,,False,,,NKG
277,VIDRIO PARABRISAS OR2,,,False,,,OR2
278,VIDRIO PARABRISAS SIN SENSOR LLUVIA NDU,,,False,,,NDU
279,VIDRIO PTA TR IZ SA3,,,False,,,SA3
280,VIDRIO PUERTA DELANTERA DERECHA NDUH,,,False,,,NDUH


### Filas de pack ambiguo (revisar a mano)

Tienen expresión de pack pero sin "L" explícito, así que no se pudo separar litraje de multiplicador de caja - quedaron intactas en `nombre_limpio`. ¿`4X4 UNID` es un típo de `4LX4UND`? ¿Qué significan `3X5` / `6X1` de Castrol?

In [6]:
df_norm[df_norm["pack_ambiguo"]][["nombre_original", "nombre_limpio", "pack"]]

,nombre_original,nombre_limpio,pack
2,10W30 MOTRIO 4X4 UNID,10W30 MOTRIO 4X4 UND,4X4 UND
24,AMOR TRA MOT 4X4 DU,AMORTIGUADOR TRASERO MOTOR 4X4 DU,4X4
65,CASTROL 10W40 3X5,CASTROL 10W40 3X5,3X5
66,CASTROL 10W40 6X1,CASTROL 10W40 6X1,6X1
67,CASTROL 10W40 6X1,CASTROL 10W40 6X1,6X1
68,CASTROL 15W40 3X5,CASTROL 15W40 3X5,3X5
69,CASTROL 15W40 3X5,CASTROL 15W40 3X5,3X5
70,CASTROL 15W40 6X1,CASTROL 15W40 6X1,6X1
71,CASTROL 15W40 6X1,CASTROL 15W40 6X1,6X1
72,CASTROL 20W50 3X5,CASTROL 20W50 3X5,3X5


## Etapa 2 — Categorización de productos, con base en los nombres


In [7]:
from pipeline.categorize import categorize_df, coverage_report
coverage_report(df_norm)          # prints per-category counts + the `otros` backlog list
cd = categorize_df(df_norm)       # adds categoria / cse_profile / presentacion columns
cd[["nombre_limpio", "categoria", "cse_profile"]].head(10)

Cobertura: 281 filas, 15 categorias

   57  carroceria
   38  vidrios_espejos
   33  iluminacion
   29  lubricantes
   21  suspension_direccion
   20  merchandising
   17  frenos
   13  motor_transmision
   13  emblemas
   10  filtros
    9  refrigeracion
    7  llantas_rines
    7  baterias
    4  otros
    3  accesorios

'otros' (backlog para nuevas reglas): 4 (1%)
    - KIT JUNTA DECANTADOR NT2
    - KIT JUNTAS INDICADOR CARBURANTE
    - MODULO VARIA CLIM NM
    - PLASTIGLO 250ML TT


,nombre_limpio,categoria,cse_profile
0,10W30 MOTRIO 1L,lubricantes,baseline
1,10W30 MOTRIO 4L,lubricantes,baseline
2,10W30 MOTRIO 4X4 UND,lubricantes,baseline
3,10W40 MOTRIO A3/B4 SS 1L,lubricantes,baseline
4,10W40 MOTRIO A3/B4 SS 4L,lubricantes,baseline
5,15W40 MOTRIO 1L,lubricantes,baseline
6,15W40 MOTRIO 4L,lubricantes,baseline
7,15W40 MOTRIO 4L,lubricantes,baseline
8,185/65R15 ENERGY XM2,llantas_rines,baseline
9,20W50 MOTRIO 1L,lubricantes,baseline


## Etapa 3 — Búsqueda CSE

Una llamada por producto. **Rung 1 (`"{ref}" "{marca}"`, ambas comilladas)** es el default: la query mínima y probada. La respuesta cruda del CSE se cachea en `cache/cse/{ref}/rung1.json`, así que iterar el resto del pipeline **no vuelve a pagar** búsquedas.

Perfil por categoría → params de imagen: `baseline` = `imgType=photo,imgSize=large`; `white_dominant` añade `imgDominantColor=white`; `exact_brand` añade `exactTerms={marca}`. Sesgo de mercado `gl=co&hl=es`.

Los rungs 2–5 (sin comillas / `nombre_limpio` / etc.) existen en `pipeline/search.py` pero **no** se auto-escalan todavía

In [8]:
# Golden set: ~1-2 productos por categoria para probar la busqueda sin gastar
# en las 281 filas. Se amplia/ajusta a mano segun lo que se quiera revisar.
golden = cd.groupby("categoria", group_keys=False).head(2).reset_index(drop=True)

print(f"Golden set: {len(golden)} productos, {golden['categoria'].nunique()} categorias")
golden[["Ref Proveedor", "Marca", "nombre_limpio", "categoria", "cse_profile"]]


Golden set: 30 productos, 15 categorias


,Ref Proveedor,Marca,nombre_limpio,categoria,cse_profile
0,8550504748,MOTRIO,10W30 MOTRIO 1L,lubricantes,baseline
1,8550504747,MOTRIO,10W30 MOTRIO 4L,lubricantes,baseline
2,7711649209,RENAULT,185/65R15 ENERGY XM2,llantas_rines,baseline
3,631019973R,RENAULT,ALETA DELANTERO IZQUIERDO SA3-LN3,carroceria,baseline
4,631000689R,RENAULT,ALETA DELANTERO DERECHO CP,carroceria,baseline
5,8660005993,RENAULT,AMORTIGUADOR TRASERO MOTOR 4X4 DU,suspension_direccion,baseline
6,543022720R,RENAULT,AMORTIGUADOR DELANTERO OR,suspension_direccion,baseline
7,D40604JA0A,RENAULT,BANDAS FRENO TRASERO ALASKAN,frenos,baseline
8,03-T4770,RENAULT,BATERIA,baterias,baseline
9,244103090R,RENAULT,BATERIA 14AH TWIZY,baterias,baseline


In [9]:
# rung = cual query construir:  1 -> "ref" "marca" (default) | 2 -> ref marca | 3 -> nombre_limpio
# max_queries = tope de llamadas NUEVAS a la API (protege las 100 gratis/dia).
# La respuesta cruda queda en cache/cse/{ref}/rung{n}.json -> re-correr no re-paga.
from pipeline.search import search_df

resultados = search_df(golden, rung=1, max_queries=40)

# Resumen: cuantos candidatos trajo cada producto y de que dominios
resumen = pd.DataFrame([
    {"ref": ref,
     "n_candidatos": len(cands),
     "dominios": ", ".join(sorted({c["displayLink"] for c in cands})[:5])}
    for ref, cands in resultados.items()
])
resumen


rung 1: 30 queries nuevas a la API (cap 40), 0 desde cache, 0 omitidas por el cap


,ref,n_candidatos,dominios
0,8550504748,7,"motrio.com.co, www.imotriz.com"
1,8550504747,6,"motrio.com.co, www.imotriz.com"
2,7711649209,10,"listado.mercadolibre.com.co, listado.tucarro.com.co"
3,631019973R,10,"avtopro.es, partsouq.com, www.b-parts.com, www.dts.lv, w..."
4,631000689R,10,"boodmo.com, loja.rpoint.com.br, podkapotom.ru, www.b-par..."
5,8660005993,10,"avto.pro, avtopro.es, www.autodoc.parts, www.autodoc24.f..."
6,543022720R,10,"loja.grupoamazonas.com.br, www.jotacarautopecas.com.br, ..."
7,D40604JA0A,10,"club.autodoc.co.uk, www.autodoc.parts, www.ebay.co.uk, w..."
8,03-T4770,0,
9,244103090R,10,"baterias.com, www.autodoc.parts, www.b-parts.com, www.de..."


In [10]:
# Preview inline: miniaturas por producto para revisar a ojo el pool crudo
from IPython.display import HTML, display

for ref, cands in resultados.items():
    fila = golden[golden["Ref Proveedor"] == ref].iloc[0]
    print(f"{ref}  |  {fila['nombre_limpio']}  |  {fila['categoria']} "
        f"({fila['cse_profile']})  |  {len(cands)} candidatos")
    imgs = "".join(
        f'<img src="{c["thumbnailLink"]}" title="{c["displayLink"]}" '
        f'style="height:90px;margin:2px;border:1px solid #ccc">'
        for c in cands if c["thumbnailLink"]
    )
    display(HTML(imgs or "<i>(sin candidatos)</i>"))


8550504748  |  10W30 MOTRIO 1L  |  lubricantes (baseline)  |  7 candidatos


8550504747  |  10W30 MOTRIO 4L  |  lubricantes (baseline)  |  6 candidatos


7711649209  |  185/65R15 ENERGY XM2  |  llantas_rines (baseline)  |  10 candidatos


631019973R  |  ALETA DELANTERO IZQUIERDO SA3-LN3  |  carroceria (baseline)  |  10 candidatos


631000689R  |  ALETA DELANTERO DERECHO CP  |  carroceria (baseline)  |  10 candidatos


...

8671002274  |  FILTRO ACEITE MOTRI TT  |  filtros (baseline)  |  10 candidatos


118301718R  |  KIT JUNTA DECANTADOR NT2  |  otros (baseline)  |  10 candidatos


7701207449  |  KIT JUNTAS INDICADOR CARBURANTE  |  otros (baseline)  |  10 candidatos


749M65735R  |  TAPETE TERMOFORMADO MILDH NDUH  |  accesorios (baseline)  |  10 candidatos


749M65098R  |  TAPETE TERMOFORMADO NDUH  |  accesorios (baseline)  |  10 candidatos


## Etapa 4 — Pre-filtro (sin Gemini)

Sobre el pool crudo de la Etapa 3: dedupe por URL y por contenido visual (perceptual hash —> misma foto servida desde dos dominios distintos), descarta resolución baja / aspect ratio de banner / dominios en `config/domains.py`, y luego **ordena** (no descarta) por dominio prioritario + score de fondo blanco calculado del thumbnail. Los primeros `keep=6` que sobreviven se verifican con un HEAD (sin bajar la imagen completa —> Gemini la pedirá después, en memoria).

Todo el proceso escribe **solo un JSON de resultado por producto** en `cache/prefilter/{ref}.json` — no se guardan imágenes a disco, así que re-correr esta celda no vuelve a pegarle a la red.

In [11]:
# Corre el pre-filtro sobre el pool de la Etapa 3 (golden set). use_cache=True
# por defecto en prefilter_product: re-correr esta celda no re-hace los HEAD
# ni vuelve a bajar thumbnails, sirve el JSON de cache/prefilter/{ref}.json.
from pipeline.prefilter import prefilter_product

filtrados = {ref: prefilter_product(ref, cands) for ref, cands in resultados.items()}

resumen_prefiltro = pd.DataFrame([
    {"ref": ref,
     "n_input": r["n_input"],
     "n_kept": len(r["kept"]),
     "n_dropped": len(r["dropped"])}
    for ref, r in filtrados.items()
])
resumen_prefiltro


,ref,n_input,n_kept,n_dropped
0,8550504748,7,2,5
1,8550504747,6,2,4
2,7711649209,10,0,10
3,631019973R,10,5,3
4,631000689R,10,4,6
5,8660005993,10,6,4
6,543022720R,10,6,2
7,D40604JA0A,10,6,4
8,03-T4770,0,0,0
9,244103090R,10,2,8


In [12]:
# Preview inline: candidatos sobrevivientes (con su bg_score) y por que se
# cayeron los demas - clave para calibrar MIN_SIDE / MAX_ASPECT_RATIO /
# HASH_MAX_DISTANCE y para ir llenando config/domains.py con evidencia real.
for ref, r in filtrados.items():
    fila = golden[golden["Ref Proveedor"] == ref].iloc[0]
    print(f"{ref}  |  {fila['nombre_limpio']}  |  {fila['categoria']}  "
        f"|  {len(r['kept'])}/{r['n_input']} sobrevivieron")

    imgs = "".join(
        f'<div style="display:inline-block;text-align:center;margin:2px">'
        f'<img src="{c["thumbnailLink"]}" title="{c["displayLink"]}" '
        f'style="height:90px;border:1px solid #ccc"><br>'
        f'<small>{c["bg_score"]}</small></div>'
        for c in r["kept"] if c["thumbnailLink"]
    )
    display(HTML(imgs or "<i>(sin candidatos sobrevivientes)</i>"))

    if r["dropped"]:
        razones = pd.Series([reason for _, reason in r["dropped"]]).value_counts()
        print("  descartados:", dict(razones))


8550504748  |  10W30 MOTRIO 1L  |  lubricantes  |  2/7 sobrevivieron


  descartados: {'resolucion_baja 300x205': np.int64(5)}
8550504747  |  10W30 MOTRIO 4L  |  lubricantes  |  2/6 sobrevivieron


  descartados: {'resolucion_baja 300x205': np.int64(4)}
7711649209  |  185/65R15 ENERGY XM2  |  llantas_rines  |  0/10 sobrevivieron


  descartados: {'resolucion_baja 320x320': np.int64(10)}
631019973R  |  ALETA DELANTERO IZQUIERDO SA3-LN3  |  carroceria  |  5/10 sobrevivieron


  descartados: {'resolucion_baja 500x375': np.int64(2), 'resolucion_baja 340x227': np.int64(1)}
631000689R  |  ALETA DELANTERO DERECHO CP  |  carroceria  |  4/10 sobrevivieron


...

  descartados: {'resolucion_baja 220x220': np.int64(1), 'resolucion_baja 379x400': np.int64(1), 'resolucion_baja 357x400': np.int64(1)}
118301718R  |  KIT JUNTA DECANTADOR NT2  |  otros  |  4/10 sobrevivieron


  descartados: {'resolucion_baja 390x295': np.int64(1), 'duplicado_visual': np.int64(1)}
7701207449  |  KIT JUNTAS INDICADOR CARBURANTE  |  otros  |  4/10 sobrevivieron


  descartados: {'duplicado_visual': np.int64(2), 'resolucion_baja 480x357': np.int64(1), 'resolucion_baja 350x291': np.int64(1)}
749M65735R  |  TAPETE TERMOFORMADO MILDH NDUH  |  accesorios  |  5/10 sobrevivieron


  descartados: {'resolucion_baja 480x360': np.int64(2), 'resolucion_baja 528x396': np.int64(1), 'resolucion_baja 228x228': np.int64(1), 'resolucion_baja 400x264': np.int64(1)}
749M65098R  |  TAPETE TERMOFORMADO NDUH  |  accesorios  |  5/10 sobrevivieron


  descartados: {'resolucion_baja 228x228': np.int64(1), 'resolucion_baja 400x264': np.int64(1), 'resolucion_baja 500x347': np.int64(1), 'resolucion_baja 500x351': np.int64(1)}


## Etapa 5 — Selección con Gemini (Vertex AI)

Una llamada por producto: las candidatas que sobrevivieron el pre-filtro (columna `kept` de la Etapa 4), numeradas, más un prompt que pide JSON estructurado (`response_mime_type="application/json"` + `response_schema`, mismos campos que el plan: `evaluaciones`, `seleccion`, `ranking`, `confianza`, `comentario`). Gemini evalúa criterios eliminatorios (producto correcto, marca) y de calidad (resolución, fondo, estado, empaque, limpieza), y ELIGE la mejor (se plantea un código a utilizar sobre elección en los casos borde).

Modelo por defecto: `gemini-2.5-flash` (barato, discriminación no generación). `MODEL_PRO` está declarado en `pipeline/select.py` para el ruteo a dos niveles (confianza baja → Pro) pero ese ruteo automático **no** está implementado todavía —> se pasa `model=` explícito si se quiere probar.

Reglas post-Gemini ya aplicadas dentro de `select_product`:
- `seleccion` fuera de rango → se resuelve al siguiente número del propio `ranking`, sin nueva llamada.
- marca de la imagen elegida = `no_verificable` → flag `revisar_marca`.
- `seleccion=null` o `confianza=baja` → flag `necesita_fallback` (la Etapa 5b la re-dispara automáticamente por los rungs 2–3).

Cache en `cache/select/{ref}.json` —> re-correr esta celda no vuelve a pagar la llamada a Gemini. Requiere `credentials.json` + `GOOGLE_CLOUD_PROJECT`/`GOOGLE_CLOUD_LOCATION` en `.env` (ya configurados).

In [21]:
# max_calls = tope de llamadas NUEVAS a Gemini (protege costo mientras se
# calibra el prompt). La respuesta ya procesada queda en cache/select/{ref}.json
# -> re-correr no vuelve a pagar.
from pipeline.select import select_df, MODEL_FLASH

seleccionados = select_df(golden, filtrados, model=MODEL_FLASH, max_calls=40)

resumen_seleccion = pd.DataFrame([
    {"ref": ref,
    "confianza": r["confianza"],
    "flags": ", ".join(r["flags"]) or "-",
    "comentario": r["comentario"]}
    for ref, r in seleccionados.items()
])
resumen_seleccion


select: 0 llamadas nuevas a Gemini (cap 40), 30 desde cache, 0 omitidas por el cap


,ref,confianza,flags,comentario
0,8550504748,alta,-,La imagen 1 es una representación perfecta del producto ...
1,8550504747,alta,-,La imagen 1 fue descartada por no mostrar el producto ni...
2,7711649209,alta,-,"Las imágenes 3 y 4 son las mejores opciones, ambas prese..."
3,631019973R,media,revisar_marca,Ninguna imagen muestra evidencia de la marca RENAULT. La...
4,631000689R,alta,revisar_marca,Todas las imágenes muestran correctamente una aleta dela...
5,8660005993,alta,revisar_marca,"Las imágenes 1, 4 y 5 fueron descartadas por mostrar mar..."
6,543022720R,media,revisar_marca,Todas las imágenes muestran el producto correcto (amorti...
7,D40604JA0A,alta,revisar_marca,"Las imágenes 2, 3 y 4 fueron descartadas por no mostrar ..."
8,03-T4770,alta,revisar_marca,Las imágenes 1 a 5 fueron descartadas porque muestran se...
9,244103090R,alta,-,La Imagen 1 es seleccionada porque muestra la marca requ...


In [22]:
# Preview inline: candidato elegido (borde verde) + el resto del pool con la
# razon de Gemini si lo descarto. Flags a revisar a mano:
#   revisar_marca      -> la marca de la elegida no se pudo verificar
#   necesita_fallback  -> seleccion=null o confianza baja (siguiente escalon
#                          de busqueda de la Etapa 3, todavia manual)
for ref, r in seleccionados.items():
    fila = golden[golden["Ref Proveedor"] == ref].iloc[0]
    sel = r["seleccion"]
    kept = filtrados.get(ref, {}).get("kept", [])
    evals = {e["imagen"]: e for e in r["evaluaciones"]}

    print(f"{ref}  |  {fila['nombre_limpio']}  |  {fila['categoria']}  "
        f"|  confianza={r['confianza']}  |  flags={', '.join(r['flags']) or '-'}")
    print(f"  {r['comentario']}")

    imgs = ""
    for i, c in enumerate(kept, start=1):
        is_sel = sel is not None and c.get("link") == sel.get("link")
        e = evals.get(i, {})
        label = "ELEGIDA" if is_sel else (e.get("razon", "") if e.get("descartada") else "")
        imgs += (
            f'<div style="display:inline-block;text-align:center;margin:3px;max-width:110px">'
            f'<img src="{c["thumbnailLink"]}" title="{c["displayLink"]}" '
            f'style="height:90px;border:{"3px solid #2a2" if is_sel else "1px solid #ccc"}"><br>'
            f'<small>{label}</small></div>'
        )
    display(HTML(imgs or "<i>(sin candidatos evaluados)</i>"))


8550504748  |  10W30 MOTRIO 1L  |  lubricantes  |  confianza=alta  |  flags=-
  La imagen 1 es una representación perfecta del producto 10W30 MOTRIO 1L, incluyendo la equivalencia de 0.946L aceptada. La imagen 2 no corresponde al producto ni a la marca.


8550504747  |  10W30 MOTRIO 4L  |  lubricantes  |  confianza=alta  |  flags=-
  La imagen 1 fue descartada por no mostrar el producto ni la marca correctos. La imagen 2 muestra el lubricante 10W30 MOTRIO en un envase de 1 galón (3.785L), lo cual es aceptable como equivalente a 4L según las reglas de presentación. La marca es visible y la calidad de la imagen es muy buena.


7711649209  |  185/65R15 ENERGY XM2  |  llantas_rines  |  confianza=alta  |  flags=-
  Las imágenes 3 y 4 son las mejores opciones, ambas presentan el producto correcto con un fondo blanco limpio y alta resolución. La imagen 1 también es correcta pero tiene un fondo monocromático en lugar de blanco. Las demás imágenes fueron descartadas por no mostrar el producto correcto o por tener marcas de agua.


631019973R  |  ALETA DELANTERO IZQUIERDO SA3-LN3  |  carroceria  |  confianza=media  |  flags=revisar_marca
  Ninguna imagen muestra evidencia de la marca RENAULT. La imagen 2 fue descartada por no mostrar el producto completo. Las imágenes 4 y 5 tienen la puntuación de calidad más alta, pero ambas presentan marcas de agua extensas en el fondo, lo que reduce la puntuación de limpieza. La imagen 1 también tiene una marca de agua prominente. Se seleccionó la imagen 4 por su alta resolución y buen estado del producto, a pesar de las marcas de agua.


631000689R  |  ALETA DELANTERO DERECHO CP  |  carroceria  |  confianza=alta  |  flags=revisar_marca
  Todas las imágenes muestran correctamente una aleta delantera individual. Ninguna imagen presenta evidencia verificable de la marca RENAULT, siendo todas 'no_verificable'. La imagen 2 fue seleccionada por tener la puntuación de calidad más alta, destacando por su fondo blanco puro, excelente resolución, estado impecable y limpieza total.


...

8671002274  |  FILTRO ACEITE MOTRI TT  |  filtros  |  confianza=alta  |  flags=-
  La imagen 2 es la mejor opción. Presenta el producto y su caja con la marca verificada, tiene alta resolución, un fondo blanco limpio y el producto en perfecto estado. Las imágenes 3 y 4 también son buenas, pero su fondo no es completamente blanco. La imagen 1 fue penalizada por la marca de agua y la imagen 6 por el fondo desordenado. La imagen 5 fue descartada por no mostrar el producto, sino una etiqueta.


118301718R  |  KIT JUNTA DECANTADOR NT2  |  otros  |  confianza=alta  |  flags=sin_imagen, necesita_fallback
  Ninguna de las imágenes proporcionadas cumple con los criterios eliminatorios. Todas muestran productos incorrectos y/o marcas diferentes a RENAULT.


7701207449  |  KIT JUNTAS INDICADOR CARBURANTE  |  otros  |  confianza=alta  |  flags=revisar_marca
  Solo la imagen 3 cumple con los criterios eliminatorios (producto correcto y marca no_verificable) y tiene la puntuación de calidad más alta entre las que pasaron.


749M65735R  |  TAPETE TERMOFORMADO MILDH NDUH  |  accesorios  |  confianza=alta  |  flags=-
  Image 4 provides the best resolution and clarity for the product and brand engraving, while adhering to all eliminatory and quality criteria. Images 1 and 3 are also valid but score slightly lower on resolution. Images 2 and 5 are discarded due to presentation rule violation and a watermark, respectively.


749M65098R  |  TAPETE TERMOFORMADO NDUH  |  accesorios  |  confianza=media  |  flags=revisar_marca
  Solo la imagen 3 muestra el producto correcto (tapetes termoformados) y cumple con los criterios eliminatorios. Sin embargo, presenta deficiencias en la calidad de la imagen, como un fondo desordenado (piso de baldosas) y una marca de agua prominente, además de que la marca RENAULT no es directamente verificable en el producto.


## Etapa 5b — Fallback (rungs 2–3)

Re-ataca SOLO los productos que salieron de la Etapa 5 con `sin_imagen` / `necesita_fallback`: baja por la escalera de búsqueda (`rung 2` = ref+marca sin comillas, `rung 3` = `nombre_limpio`), fusiona los candidatos nuevos con el pool que ya había (dedupe por link), **excluye** las imágenes que un Gemini anterior condenó por criterio eliminatorio (producto incorrecto / marca incorrecta —> no se re-pagan ni ocupan los 6 cupos del pre-filtro; las descartadas solo por calidad o con marca `no_verificable` sí se quedan, pueden ganar contra un pool más débil), re-corre pre-filtro + selección Gemini, y se detiene en el primer rung que resuelva. `max_queries` / `max_calls` protegen los presupuestos igual que en las Etapas 3 y 5.

El resultado escalado queda en `cache/fallback/{ref}.json` —> re-correr esta celda no vuelve a pagar. `escalate_df` MUTA `resultados` / `filtrados` / `seleccionados` en su lugar, así que las celdas de la Etapa 6 exportan el pipeline ya rescatado sin cambios. El log retornado (una fila por ref+rung intentado) es la métrica de cuánto rescató cada escalón extra.

In [23]:
# Escalado automatico para los productos flagged tras la Etapa 5. El log
# muestra que rung rescato que producto (o donde sigue perdido).
from pipeline.fallback import escalate_df

log_fallback = escalate_df(golden, resultados, filtrados, seleccionados,
                           max_queries=20, max_calls=20)
log_fallback

fallback: 2 productos a escalar (rungs [2, 3])
fallback: 0/2 productos rescatados, 0 queries CSE nuevas, 0 llamadas Gemini nuevas


,ref,rung,n_nuevos,n_excluidas,resultado,n_kept,confianza,flags
0,908127124R,2,0,6,sin_candidatos_nuevos,NaN,NaN,NaN
1,908127124R,3,9,6,sigue_pendiente,6.0,alta,"sin_imagen, necesita_fallback"
2,118301718R,2,0,4,sin_candidatos_nuevos,NaN,NaN,NaN
3,118301718R,3,10,4,sigue_pendiente,3.0,alta,"sin_imagen, necesita_fallback"


## Métricas — cobertura y costo

Dos vistas sobre el mismo run:

- **Cobertura** (`selection_report`): embudo por producto (candidatos CSE → sobreviven pre-filtro → evaluadas por Gemini → seleccionada) + tasa de éxito, confianza y flags. La línea "sin imagen por etapa" dice DÓNDE se pierde cada producto —> si el pool CSE está vacío, el fix es la búsqueda (rungs/perfiles); si Gemini descarta todo, el fix es el prompt o el pre-filtro.
- **Costo** (`cost_report` / `retry_report`): leídos del ledger `cache/metrics/events.jsonl`, donde cada llamada NUEVA (no cacheada) a CSE/Gemini y cada reintento escriben una línea. Los cache-hits no cuestan y no se registran. Los tokens de Gemini vienen de `usage_metadata` (el thinking se factura como salida) y quedan también dentro de `cache/select/{ref}.json` (`usage`, `modelo`, `intentos`).

Nota: los `cache/select/{ref}.json` creados ANTES de esta instrumentación no traen tokens; para backfillear, borrar `cache/select/` y re-correr la Etapa 5 (se re-paga la llamada).

In [24]:
# Embudo de cobertura por producto + resumen agregado (tasa de exito,
# confianza, flags, y en que etapa se pierden los productos sin imagen).
from pipeline.metrics import selection_report

por_ref, resumen = selection_report(golden, resultados, filtrados, seleccionados)
por_ref

28/30 productos con imagen (93%)
  confianza: {'alta': 27, 'media': 3}
  flags: {'revisar_marca': 16, 'sin_imagen': 2, 'necesita_fallback': 2}
  sin imagen por etapa: CSE vacio=0, prefiltro vacio=0, Gemini descarto todo=2


,ref,n_cse,n_prefiltro,n_evaluadas,seleccionada,confianza,flags
0,8550504748,7,2,2,True,alta,-
1,8550504747,6,2,2,True,alta,-
2,7711649209,20,6,6,True,alta,-
3,631019973R,10,5,5,True,media,revisar_marca
4,631000689R,10,4,4,True,alta,revisar_marca
5,8660005993,10,6,6,True,alta,revisar_marca
6,543022720R,10,6,6,True,media,revisar_marca
7,D40604JA0A,10,6,6,True,alta,revisar_marca
8,03-T4770,20,6,6,True,alta,revisar_marca
9,244103090R,10,2,2,True,alta,-


In [25]:
# Gasto real acumulado del ledger (queries CSE facturables por dia + tokens
# Gemini a precios de lista) y reintentos por error transitorio.
from pipeline.metrics import cost_report, retry_report

display(cost_report())
retry_report()

Costo total estimado: $0.2975 USD (41 queries CSE, 38 llamadas Gemini)


,servicio,llamadas,tokens_entrada,tokens_salida,costo_usd,nota
0,CSE,41,NaN,NaN,0.0000,0 facturables tras 100 gratis/dia
1,gemini-2.5-flash,38,67051.0,110937.0,0.2975,salida incluye thinking tokens


Sin reintentos registrados


,kind,error,n


## Etapa 6 — Salida a Excel

Copia el catálogo original y le agrega las columnas del plan: `imagen_url`, `imagen_local`, `categoria`, `termino_busqueda`, `perfil_cse_usado`, `intento_cse`, `query_cse`, `pasadas_gemini`, `reintentos_api_gemini`, `confianza`, `flags`, `razon_gemini`. La imagen que eligió Gemini se descarga a disco en `output/images/{ref}.jpg` y se agrega como miniatura real en una columna nueva `miniatura` del `.xlsx` final, para revisar el catálogo completo sin abrir una sola URL.

Nota de flags: el plan menciona `fallback_usado`, pero el pipeline emite `necesita_fallback`; la Etapa 5b ya re-dispara automáticamente los rungs 2–3, así que un producto que llega aquí con ese flag agotó la escalera —> la columna `flags` refleja tal cual lo que trae `seleccionados`, sin renombrar.

`output/` está en `.gitignore`

In [26]:
# download_images=True baja a disco la imagen elegida por producto
# (output/images/{ref}.jpg), cacheada -> re-correr no vuelve a descargar.
from pipeline.io_excel import build_output_df

out_df = build_output_df(golden, seleccionados)
out_df[["Ref Proveedor", "Marca", "categoria", "confianza", "flags", "intento_cse", "query_cse", "pasadas_gemini", "reintentos_api_gemini", "imagen_url", "imagen_local"]]


,Ref Proveedor,Marca,categoria,categoria,confianza,flags,imagen_url,imagen_local
0,8550504748,MOTRIO,lubricantes,lubricantes,alta,,https://motrio.com.co/wp-content/uploads/2023/12/MOTRIO_...,/Users/nataliavillegas/Documents/FUTURE/DONREP/image_pro...
1,8550504747,MOTRIO,lubricantes,lubricantes,alta,,https://motrio.com.co/wp-content/uploads/2023/12/MOTRIO_...,/Users/nataliavillegas/Documents/FUTURE/DONREP/image_pro...
2,7711649209,RENAULT,llantas_rines,llantas_rines,alta,,https://e-catalog.com/jpg_zoom1/187506.jpg,/Users/nataliavillegas/Documents/FUTURE/DONREP/image_pro...
3,631019973R,RENAULT,carroceria,carroceria,media,revisar_marca,https://dts.lv/media/catalog_images/631019973R_202601281...,/Users/nataliavillegas/Documents/FUTURE/DONREP/image_pro...
4,631000689R,RENAULT,carroceria,carroceria,alta,revisar_marca,https://webcdn.intercars.eu/files/424106/6504-04-6009312...,/Users/nataliavillegas/Documents/FUTURE/DONREP/image_pro...
5,8660005993,RENAULT,suspension_direccion,suspension_direccion,alta,revisar_marca,https://cdn.autodoc.de/thumb?id=24479314&m=0&n=6&lng=en&...,/Users/nataliavillegas/Documents/FUTURE/DONREP/image_pro...
6,543022720R,RENAULT,suspension_direccion,suspension_direccion,media,revisar_marca,https://amazonas-maverick-produtos.s3.amazonaws.com/loja...,/Users/nataliavillegas/Documents/FUTURE/DONREP/image_pro...
7,D40604JA0A,RENAULT,frenos,frenos,alta,revisar_marca,https://i.ebayimg.com/images/g/lOAAAOSwF6pnElyt/s-l1200.jpg,/Users/nataliavillegas/Documents/FUTURE/DONREP/image_pro...
8,03-T4770,RENAULT,baterias,baterias,alta,revisar_marca,https://www.zonagarden.com/11804-medium_default/bateria-...,/Users/nataliavillegas/Documents/FUTURE/DONREP/image_pro...
9,244103090R,RENAULT,baterias,baterias,alta,,https://www.der-ersatzteile-profi.de/images/products/big...,/Users/nataliavillegas/Documents/FUTURE/DONREP/image_pro...


In [27]:
# Escribe el xlsx y coloca la miniatura de cada imagen_local en una
# columna nueva ("miniatura") -> abrir output/catalogo_con_imagenes.xlsx
# para revisar el golden set completo sin salir de Excel.
from pipeline.io_excel import write_excel

OUTPUT_XLSX = ROOT / "output" / "catalogo_con_imagenes.xlsx"
write_excel(out_df, OUTPUT_XLSX)


excel: /Users/nataliavillegas/Documents/FUTURE/DONREP/image_processor_matcher/output/catalogo_con_imagenes.xlsx (30 filas, 28 miniaturas incrustadas)


PosixPath('/Users/nataliavillegas/Documents/FUTURE/DONREP/image_processor_matcher/output/catalogo_con_imagenes.xlsx')